In [7]:
import pandas as pd
from pathlib import Path
gtf_path = Path('..') / 'data' / 'gencode.v49.primary_assembly.annotation.gtf'
df = pd.read_csv(gtf_path, sep='	', comment='#', header=None,
                 names=['seqname','source','feature','start','end','score','strand','frame','attribute'])
df

Reading ../data/gencode.v49.primary_assembly.annotation.gtf


,seqname,source,feature,start,end,score,strand,frame,attribute
0,chr1,HAVANA,gene,11121,24894,.,+,.,"gene_id ""ENSG00000290825.2""; gene_type ""lncRNA..."
1,chr1,HAVANA,transcript,11121,14413,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
2,chr1,HAVANA,exon,11121,11211,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
3,chr1,HAVANA,exon,12010,12227,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
4,chr1,HAVANA,exon,12613,12721,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
...,...,...,...,...,...,...,...,...,...
7762311,KI270753.1,HAVANA,exon,44107,44491,.,+,.,"gene_id ""ENSG00000297844.1""; transcript_id ""EN..."
7762312,KI270755.1,HAVANA,gene,26350,27723,.,+,.,"gene_id ""ENSG00000309258.1""; gene_type ""lncRNA..."
7762313,KI270755.1,HAVANA,transcript,26350,27723,.,+,.,"gene_id ""ENSG00000309258.1""; transcript_id ""EN..."
7762314,KI270755.1,HAVANA,exon,26350,26743,.,+,.,"gene_id ""ENSG00000309258.1""; transcript_id ""EN..."


In [ ]:
UTRs = df[(df['feature'] == 'UTR')]

UTRs

,seqname,source,feature,start,end,score,strand,frame,attribute
2495,chr1,HAVANA,UTR,65419,65433,.,+,.,"gene_id ""ENSG00000186092.7""; transcript_id ""EN..."
2496,chr1,HAVANA,UTR,65520,65564,.,+,.,"gene_id ""ENSG00000186092.7""; transcript_id ""EN..."
2497,chr1,HAVANA,UTR,70006,71585,.,+,.,"gene_id ""ENSG00000186092.7""; transcript_id ""EN..."
6563,chr1,HAVANA,UTR,450740,450742,.,-,.,"gene_id ""ENSG00000284733.2""; transcript_id ""EN..."
7178,chr1,HAVANA,UTR,685716,685718,.,-,.,"gene_id ""ENSG00000284662.2""; transcript_id ""EN..."
...,...,...,...,...,...,...,...,...,...
7758729,KI270734.1,ENSEMBL,UTR,156447,156497,.,-,.,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."
7758730,KI270734.1,ENSEMBL,UTR,138082,138482,.,-,.,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."
7758763,KI270734.1,ENSEMBL,UTR,161689,161852,.,-,.,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."
7758764,KI270734.1,ENSEMBL,UTR,161587,161626,.,-,.,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."


In [12]:
CDS = df[(df['feature'] == 'CDS')]
CDS

,seqname,source,feature,start,end,score,strand,frame,attribute
2490,chr1,HAVANA,CDS,65565,65573,.,+,0,"gene_id ""ENSG00000186092.7""; transcript_id ""EN..."
2493,chr1,HAVANA,CDS,69037,70005,.,+,0,"gene_id ""ENSG00000186092.7""; transcript_id ""EN..."
6560,chr1,HAVANA,CDS,450743,451678,.,-,0,"gene_id ""ENSG00000284733.2""; transcript_id ""EN..."
7175,chr1,HAVANA,CDS,685719,686654,.,-,0,"gene_id ""ENSG00000284662.2""; transcript_id ""EN..."
8433,chr1,HAVANA,CDS,924432,924948,.,+,0,"gene_id ""ENSG00000187634.14""; transcript_id ""E..."
...,...,...,...,...,...,...,...,...,...
7758753,KI270734.1,ENSEMBL,CDS,144749,144895,.,-,0,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."
7758755,KI270734.1,ENSEMBL,CDS,143614,143789,.,-,0,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."
7758757,KI270734.1,ENSEMBL,CDS,142194,142292,.,-,1,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."
7758759,KI270734.1,ENSEMBL,CDS,138743,138831,.,-,1,"gene_id ""ENSG00000277196.4""; transcript_id ""EN..."


In [17]:
def extract_gtf_attribute(series, key):
    """Extract from attribute column (for ex. gene_id) using regex 
    given a pandas Series of attribute strings and the key to extract
    """
    return series.str.extract(fr'{key} "([^"]+)"', expand=False)


def format_interval(row):
    """Format as chromosome:start-end(strand)
    """
    return f"{row['seqname']}:{row['start']}-{row['end']}({row['strand']})"


# Only keep rows that are UTR or CDS
feature_rows = df.loc[df['feature'].isin(['UTR', 'CDS'])].copy()

# Extract gene_id and transcript_id
feature_rows['gene_id'] = extract_gtf_attribute(feature_rows['attribute'], 'gene_id')
feature_rows['transcript_id'] = extract_gtf_attribute(feature_rows['attribute'], 'transcript_id')

# Drop rows that don't have both gene_id and transcript_id
### Q: Why would some rows be missing gene_id or transcript_id?
### Maybe incomplete annotations?
feature_rows = feature_rows.dropna(subset=['gene_id', 'transcript_id'])

records = []

for transcript_id, transcript in feature_rows.groupby('transcript_id', sort=False):
    cds = transcript.loc[transcript['feature'] == 'CDS']
    utrs = transcript.loc[transcript['feature'] == 'UTR']

    # If no CDS, skip because otherwise no way to tell if it's a 5' UTR or 3' UTR
    if cds.empty:
        continue

    strand = transcript['strand'].iloc[0]
    gene_id = transcript['gene_id'].iloc[0]

    cds_start = cds['start'].min()
    cds_end = cds['end'].max()

    for _, row in cds.iterrows():
        records.append({'gene_id': gene_id, 'feature': 'CDS', 'interval': format_interval(row)})

    for _, row in utrs.iterrows():
        if strand == '+':
            # UTRs before CDS are 5', after are 3'
            if row['end'] < cds_start:
                feature = "5' UTR"
            elif row['start'] > cds_end:
                feature = "3' UTR"
            else:
                feature = None
        else:
            # Reverse
            if row['start'] > cds_end:
                feature = "5' UTR"
            elif row['end'] < cds_start:
                feature = "3' UTR"
            else:
                feature = None
        if feature is not None:
            records.append({'gene_id': gene_id, 'feature': feature, 'interval': format_interval(row)})

    # Join intervals for the same gene
    gene_feature_df = (
        pd.DataFrame(records)
          .groupby(['gene_id', 'feature'], as_index=False)['interval']
          .agg('; '.join)  # join multiple intervals with semicolons
          .pivot(index='gene_id', columns='feature', values='interval')
          .reindex(columns=["5' UTR", 'CDS', "3' UTR"], fill_value='')
          .rename_axis(None, axis=1)
          .reset_index()
    )

gene_feature_df.head()

KeyboardInterrupt: 

In [18]:
# Build a separate gene-level table with counts for each feature type
gene_feature_counts_df = gene_feature_df[['gene_id', "5' UTR", 'CDS', "3' UTR"]].copy()

gene_feature_counts_df["5' UTR count"] = gene_feature_counts_df["5' UTR"].fillna('').apply(lambda value: 0 if value == '' else value.count(';') + 1)
gene_feature_counts_df['CDS count'] = gene_feature_counts_df['CDS'].fillna('').apply(lambda value: 0 if value == '' else value.count(';') + 1)
gene_feature_counts_df["3' UTR count"] = gene_feature_counts_df["3' UTR"].fillna('').apply(lambda value: 0 if value == '' else value.count(';') + 1)

gene_feature_counts_df = gene_feature_counts_df[['gene_id', "5' UTR count", 'CDS count', "3' UTR count"]]
gene_feature_counts_df.head()

,gene_id,5' UTR count,CDS count,3' UTR count
0,ENSG00000000938.14,94,315,29
1,ENSG00000001460.19,13,52,8
2,ENSG00000001461.20,38,161,16
3,ENSG00000004455.19,17,73,60
4,ENSG00000004487.19,49,832,96
